In [ ]:
# Invoice Information Extraction with LoRA

## 1. Setup and Environment



In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("  No GPU detected! Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU")

GPU available: True
GPU name: Tesla T4
GPU memory: 15.64 GB


In [ ]:
!pip install -q -U transformers datasets
!pip install -q seqeval evaluate
!pip install -q peft  # ← needed for LoRA

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00


## 2. Data Loading and Preparation



In [ ]:
import os

# ─── SET YOUR DATA DIRECTORY HERE ─────────────────────────────────────────────
# If you uploaded files directly to Colab:
DATA_DIR = "/content"

# If you're using Google Drive (uncomment and edit):
# DATA_DIR = "/content/drive/MyDrive/invoice_data"

TRAIN_FILE = f"{DATA_DIR}/train.json"
VAL_FILE   = f"{DATA_DIR}/val.json"
TEST_FILE  = f"{DATA_DIR}/test.json"

# Output directory
OUTPUT_DIR = "/content/invoice_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify files exist
for f in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    status = " Found" if os.path.exists(f) else "❌ NOT FOUND — check your path!"
    print(f"  {os.path.basename(f)}: {status}")

  train.json:  Found
  val.json:  Found
  test.json:  Found


## 3. Tokenization and Labeling



In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train": TRAIN_FILE,
    "val":   VAL_FILE,
    "test":  TEST_FILE,
})

print(ds)
print("\nSample record:")
print(ds["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'vendor', 'date', 'total_amount'],
        num_rows: 2733
    })
    val: Dataset({
        features: ['text', 'vendor', 'date', 'total_amount'],
        num_rows: 586
    })
    test: Dataset({
        features: ['text', 'vendor', 'date', 'total_amount'],
        num_rows: 586
    })
})

Sample record:
{'text': '2760 out9 392 221 regis crc 3245 contract research center b v b a 8 p r l business administration b 1932 zaventem geadresseerde adress e tollaan avenue du p age 101c telefax telefoon t l phone 02 720 55 94 telefax tilffax 02 725 12 09 bbc t a v dhr verbruggen industriezone wolfstee toekomstlaan 33 jgb mjv b 2410 herentals bestelling nr commande no 627 92 datum date 23 okt 92 geneve deze toferte in leder govel ca de feldyur te vermeiden inez sens szespusa reprendre cette rirbrenes tur in fecture wo don da pi xx 230 x jok xx aankoopvoorwaarden zie xxxx brief van 12 jun 91 positie hoeveelheid eenheid artikelomschrijvin

In [ ]:
import numpy as np
from datetime import datetime
from transformers import AutoTokenizer

# ─── BIO Label Schema ─────────────────────────────────────────────────────────
LABELS = ["O", "B-VENDOR", "I-VENDOR", "B-DATE", "I-DATE", "B-AMOUNT", "I-AMOUNT"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}


def find_span(text: str, target: str):
    target_lower = target.lower().strip()
    text_lower = text.lower()
    idx = text_lower.find(target_lower)
    if idx == -1:
        return None
    return (idx, idx + len(target))


def find_date_span(text: str, date_iso: str):
    try:
        dt = datetime.fromisoformat(date_iso)
    except ValueError:
        return None
    text_lower = text.lower()
    string_formats = [
        dt.strftime("%B %d %Y"), dt.strftime("%B %d, %Y"),
        dt.strftime("%b %d %Y"), dt.strftime("%b %d, %Y"),
        dt.strftime("%d %B %Y"), dt.strftime("%d %b %Y"),
        dt.strftime("%Y-%m-%d"),
    ]
    for fmt in string_formats:
        idx = text_lower.find(fmt.lower())
        if idx != -1:
            return (idx, idx + len(fmt))
    y, m, d = dt.year, dt.month, dt.day
    y2 = y % 100
    numeric_patterns = [
        f"{m:02d}/{d:02d}/{y}", f"{m}/{d}/{y}",
        f"{m:02d}-{d:02d}-{y}", f"{m}-{d}-{y}",
        f"{m:02d}.{d:02d}.{y}", f"{m}.{d}.{y}",
        f"{m:02d}/{d:02d}/{y2:02d}", f"{m}/{d}/{y2}",
        f"{m:02d}-{d:02d}-{y2:02d}", f"{m}-{d}-{y2}",
        f"{d:02d}/{m:02d}/{y}", f"{d}/{m}/{y}",
        f"{d:02d}-{m:02d}-{y}", f"{d}-{m}-{y}",
        f"{d} {m} {y}", f"{d} {m} {y2}",
        f"{d:02d} {m:02d} {y}", f"{d:02d} {m:02d} {y2:02d}",
        f"{m} {d} {y}", f"{m} {d} {y2}",
        f"{y}{m:02d}{d:02d}", f"{d:02d}{m:02d}{y2:02d}", f"{d:02d}{m:02d}{y}",
    ]
    for pat in numeric_patterns:
        idx = text.find(pat)
        if idx != -1:
            return (idx, idx + len(pat))
    return None


def find_amount_span(text: str, amount: float):
    if amount < 10:
        return None
    int_amt = int(amount)
    has_decimal = amount != int_amt
    variants = []
    if has_decimal:
        variants.append(f"{amount:.2f}")
        variants.append(f"{int_amt:,}.{int(round((amount - int_amt) * 100)):02d}")
        cents = int(round((amount - int_amt) * 100))
        variants.append(f"{int_amt} {cents:02d}")
    variants.append(f"{int_amt:,}")
    variants.append(str(int_amt))
    int_str = str(int_amt)
    if len(int_str) > 3:
        rev = int_str[::-1]
        grouped = ' '.join(rev[i:i+3] for i in range(0, len(rev), 3))
        variants.append(grouped[::-1])
    for variant in variants:
        idx = text.find(variant)
        if idx != -1:
            return (idx, idx + len(variant))
    return None


MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_and_label(example):
    text = example["text"]
    spans = []
    v = find_span(text, example["vendor"])
    if v: spans.append((v[0], v[1], "VENDOR"))
    d = find_date_span(text, example["date"])
    if d: spans.append((d[0], d[1], "DATE"))
    a = find_amount_span(text, example["total_amount"])
    if a: spans.append((a[0], a[1], "AMOUNT"))

    tokenized = tokenizer(text, truncation=True, max_length=512, return_offsets_mapping=True)
    labels = [label2id["O"]] * len(tokenized["input_ids"])

    for i, (start, end) in enumerate(tokenized["offset_mapping"]):
        if start == 0 and end == 0:
            labels[i] = -100

    for span_start, span_end, field_type in spans:
        first_token = True
        for i, (tok_start, tok_end) in enumerate(tokenized["offset_mapping"]):
            if tok_start == 0 and tok_end == 0:
                continue
            if tok_start >= span_start and tok_end <= span_end:
                labels[i] = label2id[f"B-{field_type}"] if first_token else label2id[f"I-{field_type}"]
                first_token = False

    tokenized["labels"] = labels
    tokenized.pop("offset_mapping")
    return tokenized

print(" Label schema and tokenizer ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

 Label schema and tokenizer ready.


## 4. Data Preprocessing



In [ ]:
tokenized_ds = ds.map(tokenize_and_label, remove_columns=ds["train"].column_names)
print(tokenized_ds)

# Coverage check
def count_labeled(ds_split):
    stats = {"VENDOR": 0, "DATE": 0, "AMOUNT": 0, "total": 0}
    for ex in ds_split:
        stats["total"] += 1
        labels = [l for l in ex["labels"] if l != -100]
        if label2id["B-VENDOR"] in labels: stats["VENDOR"] += 1
        if label2id["B-DATE"]   in labels: stats["DATE"]   += 1
        if label2id["B-AMOUNT"] in labels: stats["AMOUNT"] += 1
    return stats

for split in ["train", "val"]:
    s = count_labeled(tokenized_ds[split])
    n = s["total"]
    print(f"\n{split.capitalize()} ({n} examples):")
    for field in ["VENDOR", "DATE", "AMOUNT"]:
        print(f"  {field:8}: {s[field]:4d}/{n}  ({100*s[field]/n:.1f}%)")

Map:   0%|          | 0/2733 [00:00<?, ? examples/s]

Map:   0%|          | 0/586 [00:00<?, ? examples/s]

Map:   0%|          | 0/586 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2733
    })
    val: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 586
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 586
    })
})

Train (2733 examples):
  VENDOR  : 2311/2733  (84.6%)
  DATE    : 1800/2733  (65.9%)
  AMOUNT  : 2574/2733  (94.2%)

Val (586 examples):
  VENDOR  :  489/586  (83.4%)
  DATE    :  379/586  (64.7%)
  AMOUNT  :  552/586  (94.2%)


## 5. Metrics and Data Collator



In [ ]:
import evaluate
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True)
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [id2label[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
        "vendor_f1": results.get("VENDOR", {}).get("f1", 0.0),
        "date_f1":   results.get("DATE",   {}).get("f1", 0.0),
        "amount_f1": results.get("AMOUNT", {}).get("f1", 0.0),
    }

print(" Metrics function ready.")

 Metrics function ready.


## 6. Baseline Model Training



In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

baseline_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id,
)

baseline_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/baseline-checkpoints",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=2,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    fp16=True,
    warmup_ratio=0.1,
)

baseline_trainer = Trainer(
    model=baseline_model,
    args=baseline_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["val"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting BASELINE full fine-tune...")
baseline_trainer.train()
print(f"Best Baseline model loaded from: {baseline_trainer.state.best_model_checkpoint}")
baseline_trainer.save_model(f"{OUTPUT_DIR}/baseline-final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/baseline-final")
print(" Baseline model saved.")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting BASELINE full fine-tune...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Vendor F1,Date F1,Amount F1
1,0.110477,0.080688,0.307438,0.261972,0.282890,0.974870,0.275648,0.483104,0.106236
2,0.060118,0.058837,0.567696,0.504930,0.534476,0.980962,0.445135,0.759331,0.440426
3,0.040907,0.057444,0.533093,0.623944,0.574951,0.980978,0.480634,0.764629,0.542056
4,0.028213,0.058432,0.570560,0.660563,0.612272,0.981892,0.500000,0.792500,0.592593
5,0.017607,0.065848,0.620902,0.640141,0.630374,0.983136,0.518738,0.807792,0.609091
6,0.011198,0.067479,0.616879,0.648592,0.632338,0.982768,0.528626,0.807198,0.607176
7,0.009766,0.075821,0.597362,0.669718,0.631474,0.982637,0.515444,0.818182,0.608108
8,0.006699,0.077793,0.609225,0.669718,0.638041,0.982837,0.524496,0.823226,0.616309
9,0.005425,0.080311,0.622654,0.654225,0.638049,0.983313,0.524655,0.824742,0.611408
10,0.004607,0.082097,0.615236,0.671127,0.641967,0.983198,0.525273,0.822023,0.622561


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best Baseline model loaded from: /content/invoice_outputs/baseline-checkpoints/checkpoint-1710


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Baseline model saved.


## 7. Baseline Model Evaluation



In [ ]:
baseline_test = baseline_trainer.predict(tokenized_ds["test"])
baseline_metrics = baseline_test.metrics
print("\n BASELINE TEST METRICS:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


 BASELINE TEST METRICS:
  test_loss: 0.0786
  test_precision: 0.6205
  test_recall: 0.6822
  test_f1: 0.6499
  test_accuracy: 0.9839
  test_vendor_f1: 0.5422
  test_date_f1: 0.8206
  test_amount_f1: 0.6207
  test_runtime: 3.9901
  test_samples_per_second: 146.8640
  test_steps_per_second: 4.7620


## 8. LoRA Model Setup



In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_base = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id,
)

lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=128,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin", "lin1", "lin2"],
    modules_to_save=["classifier"],
)

lora_model = get_peft_model(lora_base, lora_config)
lora_model.print_trainable_parameters()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 10,622,215 || all params: 76,990,478 || trainable%: 13.7968


## 9. LoRA Fine-tuning



In [ ]:
lora_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/lora-checkpoints",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=15,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=2,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    fp16=True,
    warmup_ratio=0.1,
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["val"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting LoRA fine-tuning...")
lora_trainer.train()
print(f"Best LoRA model loaded from: {lora_trainer.state.best_model_checkpoint}")
print(" LoRA training complete.")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Starting LoRA fine-tuning...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Vendor F1,Date F1,Amount F1
1,0.114407,0.082399,0.365254,0.303521,0.331538,0.974309,0.302000,0.501235,0.194937
2,0.073742,0.067803,0.514312,0.392254,0.445066,0.978680,0.354458,0.715596,0.294190
3,0.063625,0.062473,0.491492,0.528873,0.509498,0.978765,0.406615,0.701970,0.463899
4,0.051622,0.062107,0.543936,0.527465,0.535574,0.979871,0.374582,0.764187,0.502258
5,0.045192,0.059216,0.582990,0.598592,0.590688,0.981377,0.454822,0.761792,0.579904
6,0.036275,0.060469,0.588153,0.594366,0.591243,0.982022,0.456204,0.792176,0.573858
7,0.029595,0.062124,0.613619,0.564789,0.588192,0.982284,0.470588,0.788652,0.565260
8,0.023377,0.067917,0.629254,0.611972,0.620493,0.982768,0.485798,0.821382,0.604146
9,0.019948,0.069762,0.639245,0.596479,0.617122,0.982660,0.494405,0.787623,0.616556
10,0.014310,0.080098,0.636488,0.653521,0.644892,0.982752,0.512768,0.839572,0.630756


Best LoRA model loaded from: /content/invoice_outputs/lora-checkpoints/checkpoint-2565
 LoRA training complete.


## 10. Save LoRA Adapter



In [ ]:
ADAPTER_PATH = f"{OUTPUT_DIR}/lora-adapter"
lora_model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(" LoRA adapter saved.")

import os
for f in os.listdir(ADAPTER_PATH):
    size = os.path.getsize(f"{ADAPTER_PATH}/{f}")
    print(f"  {f}: {size/1024:.1f} KB")

 LoRA adapter saved.
  README.md: 5.0 KB
  tokenizer.json: 694.8 KB
  tokenizer_config.json: 0.3 KB
  adapter_model.safetensors: 41503.6 KB
  adapter_config.json: 1.1 KB


## 11. LoRA Model Evaluation


In [ ]:
lora_test = lora_trainer.predict(tokenized_ds["test"])
lora_metrics = lora_test.metrics
print("\n LORA TEST METRICS:")
for k, v in lora_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


 LORA TEST METRICS:
  test_loss: 0.0914
  test_precision: 0.6984
  test_recall: 0.6892
  test_f1: 0.6937
  test_accuracy: 0.9850
  test_vendor_f1: 0.5913
  test_date_f1: 0.8920
  test_amount_f1: 0.6420
  test_runtime: 5.1317
  test_samples_per_second: 114.1920
  test_steps_per_second: 3.7020


## 12. Download Models



In [ ]:
import shutil
from google.colab import files

# Zip the LoRA adapter (small — ~2MB)
shutil.make_archive("/content/invoice_lora_adapter", "zip", ADAPTER_PATH)
files.download("/content/invoice_lora_adapter.zip")

# Zip baseline model (large — ~260MB, may take a moment)
# shutil.make_archive("/content/invoice_baseline", "zip", f"{OUTPUT_DIR}/baseline-final")
# files.download("/content/invoice_baseline.zip")

print(" Download triggered for LoRA adapter.")
print("Uncomment the last two lines above to also download the full baseline model.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Download triggered for LoRA adapter.
Uncomment the last two lines above to also download the full baseline model.
